# AfroViewz 06: Training RF-DETR

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JBWandera/Capstone/blob/afroviewz/notebooks/06_train_rfdetr.ipynb)

**Workflow stage:** modelling. **Runtime:** T4 GPU.

One notebook runs every RF-DETR experiment in `configs/experiments.yaml`. Pick the id in the cell below and run all. A free Colab session can end at any time: everything is written to `runs/<id>/` on Drive, and running the notebook again **resumes** from the last checkpoint.

| id | model | input | purpose |
|---|---|---|---|
| E2_rfdetr_nano | RF-DETR Nano | 384 | phone and browser |
| E2_rfdetr_small | RF-DETR Small | 512 | balance |
| E2_rfdetr_medium | RF-DETR Medium | 576 | server |
| E3_rfdetr_small_704 | RF-DETR Small | 704 | does resolution fix small potholes? |

Shared settings (AdamW, cosine schedule with 1 warm-up epoch, effective batch 16, EMA weights, early stopping after 8 epochs without validation mAP gain, seed 42) are in the same file, so every run differs only in what the table says.

In [ ]:
EXP_ID = "E2_rfdetr_nano"  #@param ["E2_rfdetr_nano", "E2_rfdetr_small", "E2_rfdetr_medium", "E3_rfdetr_small_704"]

## Setup

In [ ]:
# Works on Colab and locally. On Colab: mounts Drive, gets the code, installs it.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/Capstone"):
        subprocess.run(["git", "clone", "-q", "-b", "afroviewz", "https://github.com/JBWandera/Capstone.git", "/content/Capstone"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/Capstone", "pull", "-q"], check=True)
    os.chdir("/content/Capstone")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"], check=True)
    sys.path.insert(0, "/content/Capstone/src")
from afroviewz import paths
print("data root:", paths.data_root())


In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))


In [ ]:
# Copy the dataset from Drive to the fast local disk once per session.
import zipfile, shutil
from pathlib import Path
DATASET = "afroviewz-v1"
zip_on_drive = paths.data_root() / "processed" / f"{DATASET}.zip"
LOCAL = Path("/content/datasets") if IN_COLAB else paths.processed_dir(DATASET).parent
DATASET_DIR = LOCAL / DATASET / "rfdetr"
if not (DATASET_DIR / "train" / "_annotations.coco.json").exists():
    LOCAL.mkdir(parents=True, exist_ok=True)
    shutil.copy(zip_on_drive, LOCAL / zip_on_drive.name)
    with zipfile.ZipFile(LOCAL / zip_on_drive.name) as z:
        z.extractall(LOCAL)
for s in ("train", "valid", "test"):
    n = len(list((DATASET_DIR / s).glob("*.jpg")))
    print(f"{s}: {n} images")


In [ ]:
from afroviewz.train import runner
cfg = runner.experiment_config(EXP_ID)
cfg


## Train (resumes automatically)

In [ ]:
run_dir = runner.train_rfdetr(EXP_ID, DATASET_DIR)
print("run folder:", run_dir)


## Learning curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
m = pd.read_csv(run_dir / "metrics.csv").groupby("epoch").mean(numeric_only=True).reset_index()
cols = [c for c in m.columns if "mAP" in c or c.endswith("loss")]
print(cols)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for c in [c for c in cols if c.endswith("loss")]:
    ax[0].plot(m.epoch, m[c], label=c)
ax[0].legend(); ax[0].set_title("loss")
for c in [c for c in cols if "mAP" in c]:
    ax[1].plot(m.epoch, m[c], label=c)
ax[1].legend(); ax[1].set_title("validation mAP")
plt.tight_layout(); plt.savefig(run_dir / "curves.png", dpi=120); plt.show()


## Predictions for evaluation

In [ ]:
model = runner.load_rfdetr(EXP_ID, run_dir)
for split in ("valid", "test"):
    preds, ms = runner.predict_split(runner.rfdetr_predict_fn(model), DATASET_DIR / split)
    runner.save_predictions(run_dir, split, preds, ms)
    print(split, len(preds), "predictions,", round(ms, 1), "ms per image on", torch.cuda.get_device_name(0))
